In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-


# Machine Learning

## Session 03c - Classification

###  Part 3: Santander XGBOOST

<img src='../../prasami_images/prasami_color_tutorials_small.png' style = 'width:400px;' alt="By Pramod Sharma : pramod.sharma@prasami.com" align="left"/>

In [ ]:
import sys
import os
from time import time
import numpy as np
import pandas as pd


import matplotlib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_score, 
                             recall_score, 
                             accuracy_score,
                             classification_report,
                             roc_auc_score)
import matplotlib.pyplot as plt

In [ ]:
# Some Basic Parameters for housekeeping

# set location of input files:
inpDir = os.path.join('/home/hpcap/hpcap/ai_ml_dl/directory_env', 'input')

# set location of inputs for this module
moduleDir = 'machine_learning'
# set location of output files
outDir = os.path.join('/home/hpcap/hpcap/ai_ml_dl/directory_env', 'output')

# define and set random state 
RANDOM_STATE = 24
np.random.seed(RANDOM_STATE) # Set Random Seed for reproducible  results

# parameters for Matplotlib
params = {'legend.fontsize': 'large',
          'figure.figsize': (15, 6),
          'axes.labelsize': 'large',
          'axes.titlesize':'x-large',
          'xtick.labelsize':'x-large',
          'ytick.labelsize':'large',
          'savefig.dpi': 75,
          'image.interpolation': 'none',
          'savefig.bbox' : 'tight',
          'lines.linewidth' : 1,
          'legend.numpoints' : 1
         }

CMAP = plt.cm.rainbow
plt.rcParams.update(params);
plt.set_cmap(CMAP);

plt.style.use('bmh')

TEST_SIZE = 0.05

## Load Data

In [ ]:
train_df = pd.read_csv(os.path.join(inpDir, moduleDir, '/home/hpcap/hpcap/ai_ml_dl/directory_env/santander-20230628T073640Z-001/santander/train.csv'))
test_df = pd.read_csv(os.path.join(inpDir, moduleDir, '/home/hpcap/hpcap/ai_ml_dl/directory_env/santander-20230628T073640Z-001/santander/test.csv'))
train_df.head()

In [ ]:
train_df.describe().T

In [ ]:
train_df['TARGET'].value_counts()

In [ ]:
class_names = train_df['TARGET'].unique()
class_names

In [ ]:
fig, ax = plt.subplots(figsize = (5,5))

explode = (0, 0.2)  # only "explode" the 2nd slice (i.e. '1')

ax.pie(train_df['TARGET'].value_counts(), explode=explode, 
        labels=class_names, autopct='%1.1f%%',
        shadow=True, startangle=0)

ax.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.

plt.show()

In [ ]:
# var15 is AGE
fig, axes = plt.subplots(1,2)

ax = axes[0]

train_df['var15'].hist(bins=25, ax = ax)

ax.set_title('train');

ax = axes[1]

test_df['var15'].hist(bins=25, ax = ax)

ax.set_title('test');

In [ ]:
# under 23 do not complain

print (len(train_df['TARGET'][train_df.var15<23]))

print (sum(train_df['TARGET'][train_df.var15<23]))

### Combine train and test sets

In [ ]:
## use for spltting up combined train+test data, recording predictions
labels = train_df['TARGET'].values

id_test = test_df['ID']
piv_train = train_df.shape[0]



# Creating a DataFrame with train+test data
df_all = pd.concat((train_df.drop(['TARGET'], axis=1),
                    test_df.copy()), axis=0, ignore_index=True)

# Removing id
df_all = df_all.drop(['ID'], axis=1)

print(df_all.isnull().sum() / df_all.shape[0])

df_all.head()

## Feature Engineering

In [ ]:
# Create another column with count of zeros
df_all['n0'] = (df_all==0).sum(axis=1)
print (df_all.shape)

### Remove constant columns
One can also remove constant columns by length of unique values = 1 

In [ ]:
# remove constant columns (std = 0)
drop_cols = []

for col in df_all.columns:
    
    if df_all[col].std() == 0:
        
        drop_cols.append(col)

df_all.drop(drop_cols, axis=1, inplace=True)

print (df_all.shape)

### Removal of dupe columns

In [ ]:
# keep track of columns to remove
drop_cols = []

cols = df_all.columns

# loop thru cols to find equal values
for i in range(len(cols)-1):
    
    v = df_all[cols[i]].values
    
    for j in range(i+1,len(cols)):
        
        if np.array_equal(v,df_all[cols[j]].values):
            
            drop_cols.append(cols[j])


print (len(drop_cols))
print (drop_cols)

In [ ]:
## Additional columns suggested by others

# extend the list of cols to remove
drop_cols += ['num_var13_medio', 'saldo_medio_var13_medio_ult1']

In [ ]:
df_all.drop(drop_cols, axis=1, inplace=True)

print (df_all.shape)

### Limiting min/max values

In [ ]:
# limit vars in test based on min and max vals of train
print('Setting min-max limits on test data...')

startTime = time()

for f in df_all.columns[:-1]:
    
    lim_min = np.min(train_df[f])
    
    mask = df_all[f] < lim_min
    
    df_all.loc[mask, f] = lim_min

    lim_max = np.max(train_df[f])
    
    mask = df_all[f] > lim_max
    
    df_all.loc[mask, f] = lim_max
    
    #print (f, ': min=', lim_min, ', max=', lim_max)

print (time() - startTime)

### Log Transform

In [ ]:
# Look at the data

fig, axes = plt.subplots(1,2)

axes[0].hist(np.log(train_df['var38']), bins=50)

axes[0].set_title('Training set')

axes[1].hist(np.log(test_df['var38']), bins=50)

axes[1].set_title('Test set');

In [ ]:
# log transform 'var38'
df_all['var38'] = np.log(df_all['var38'])

### Train, test and validation sets

In [ ]:
vals = df_all.copy()

fea_train_df = vals[:piv_train]
fea_test_df = vals[piv_train:]
y = labels

In [ ]:
from sklearn.model_selection import train_test_split

## split into train and validation
## use X,y for full training data
X_train, X_valid, y_train, y_valid = train_test_split(fea_train_df, y, test_size=TEST_SIZE,
                                                      stratify=y, random_state=RANDOM_STATE)

print(X_train.shape)
print(y_train.shape)

print(X_valid.shape)
print(y_valid.shape)

In [ ]:
X_train

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_valid = sc.transform(X_valid)

X_test = sc.transform(fea_test_df)

## Training

In [ ]:
from xgboost.sklearn import XGBClassifier


# set the validation set for xgb training
eval_set = [(X_train,y_train), (X_valid,y_valid)]

# try using... eval_metric: mlogloss, merror, ndcg@n-, logloss, auc
clf = XGBClassifier(eval_metric='logloss')


In [ ]:
clf.fit(X_train, y_train, 
        eval_set=eval_set, 
        early_stopping_rounds=100)

In [ ]:
y_pred = clf.predict(X_valid)

print(f'Accuracy : {accuracy_score(y_valid, y_pred):.4f}')

print(f'Precision: {precision_score(y_valid, y_pred, average="macro"):.4f}')

print(f'Recall   : {recall_score(y_valid, y_pred, average="macro"):.4f}')

In [ ]:
print(classification_report(y_valid, clf.predict(X_valid)))

In [ ]:
## xgb, predict probabilities
y_proba = clf.predict_proba(X_valid)
print (roc_auc_score(y_valid, y_proba[:,1]))

In [ ]:
### Try adjusting output of some of the parameters to zero...

# remember features that predict 0
'''zero_feats = [('saldo_medio_var5_hace2', 160000), 
              ('saldo_var33', 0),
              ('var38', 3988596),
              ('var21', 7500),
              ('num_var30', 9),
              ('num_var13_0', 6),
              ('num_var33_0', 0),
              ('imp_ent_var16_ult1', 51003),
              ('imp_op_var39_comer_ult3', 13184),
              ('saldo_medio_var5_ult3', 108251),
             ]
for x,y in zero_feats:
    print (x, train[(train[x] > y) & (train.TARGET==1)].shape)
'''
